# Multi-panel AndeanGC figure


In [ ]:
# general
import geopandas as gpd

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler

from src.utils import load_config
from src.plotting import base_layout, geo_style, add_panel_label, SEQUENTIAL_COLORSCALE, save_figure
cfg = load_config()

## Data

In [ ]:
# shapes/polygons
andeanGC_shape =  gpd.read_file(cfg["dir"]["andeangc"] + "dataset/AndeanGC_shape.gpkg").set_index("gauge_id")
andeanGC_shape = andeanGC_shape.sort_values(['basin_area', 'gauge_lat'], ascending=[False, True])

# plot elements
main_basins = gpd.read_file(cfg["dir"]["data"] + "/hydroatlas/hybas_sa_lev08_v1c_dissolved.shp")
main_basins["geometry"] = main_basins["geometry"].simplify(0.02, preserve_topology=True)
scaler = MinMaxScaler(feature_range=(7, 30))

## Plot

In [ ]:
# Create subplot structure
fig = make_subplots(
    rows=1, cols=4, 
    horizontal_spacing=0.01,
    specs=[[{"type": "scattergeo"}, {"type": "scattergeo"}, {"type": "scattergeo"}, {"type": "scattergeo"}]]
)

# Define colorscale and variables
colorscale = SEQUENTIAL_COLORSCALE

plot_config = [
    {'col': 1, 'data': andeanGC_shape.glacier_area_RGI60, 'cmin': 0, 'cmax': 20, 'title': 'Glacier<br>area<br>(%)', 'x': 0.03},
    {'col': 2, 'data': andeanGC_shape.days_w_data_qc / 365, 'cmin': 10, 'cmax': 80, 'title': 'Years<br>with<br>data (yr)', 'x': 0.29},
    {'col': 3, 'data': andeanGC_shape.prcp_mean_ERA5, 'cmin': 0, 'cmax': 3000, 'title': 'Annual<br>precipitation<br>(mm yr<sup>-1</sup>)', 'x': 0.54},
    {'col': 4, 'data': andeanGC_shape.forest_cover, 'cmin': 0, 'cmax': 80, 'title': 'Forest<br>fraction<br>(%)', 'x': 0.79},
]

# Add basin outlines
for col in range(1, 5):
    fig.add_trace(go.Choropleth(geojson=eval(main_basins['geometry'].to_json()), locations=main_basins.index, 
                                z=main_basins.NEXT_DOWN, colorscale=["rgba(255,255,255,0.0001)", "rgba(255,255,255,0.0001)"], 
                                marker_line_color = "rgb(210, 226, 230)", showscale=False, marker_line_width=0.8), 
                   row=1, col=col)

# Add scatter traces for each subplot
for config in plot_config:
    fig.add_trace(
        go.Scattergeo(
            lon=andeanGC_shape['gauge_lon'],
            lat=andeanGC_shape['gauge_lat'],
            mode='markers',
            marker=dict(
                size=scaler.fit_transform(andeanGC_shape[['basin_area']]).flatten(),
                opacity=0.7,
                color=config['data'],
                colorscale=colorscale,
                cmin=config['cmin'],
                cmax=config['cmax'],
                colorbar=dict(len=0.3, x=config['x'], y=0.65, title=config['title'], thickness=15, tickwidth=0.1, bgcolor="rgba(255,255,255, 0)"),
                line=dict(width=0.5, color='grey')
            ),
            showlegend=False
        ),
        row=1, col=config['col']
    )



# Update geography settings for each subplot
for col in range(1, 5):
    fig.update_geos(**geo_style([-79, -67]), row=1, col=col)
    # Add river labels
    fig.add_trace(
        go.Scattergeo(
            lat=[-50.0, -47.0, -42.5, -39.0, -37.0, -33.5, -36.0, -30.0, -18.0, -9.0],
            lon=[-75.5, -75.5, -74.0, -70.0, -75.0, -73.0, -68.5, -72.5, -68.0, -76.0],
            text=["Santa<br>Cruz", "Baker", "Puelo", "Negro", "Biobío", "Maipo", "Mendoza", "Elqui", "Titicaca", "Santa"],
            mode="text",
            textfont=dict(size=11, color="#5e8588"),
            showlegend=False
        ), row=1, col=col)

# Add panel labels (a, b, c, d) to upper right corner of each plot
labels = ['a)', 'b)', 'c)', 'd)']
x_positions = [0.24, 0.49, 0.74, 0.99]  # right edge of each panel with horizontal_spacing=0.01

for label, x_pos in zip(labels, x_positions):
    add_panel_label(fig, label, x=x_pos, y=0.99)

# Update layout
fig.update_layout(**base_layout(
    autosize=False, width=900, height=900, font_color="#363636",
    margin=dict(l=5, r=5, b=5, t=5),
))

save_figure(fig, "figures/fig_andeanGC_dataset.svg", width=900, height=900)
fig.show()

In [ ]:
# Add country labels
fig.add_trace(
    go.Scattergeo(
        lon=[-69.0, -64.0, -73.0, -64.0],
        lat=[-23.5, -27.0, -11.0, -18.0],
        text=["Chile", "Argentina", "Peru", "Bolivia"],
        mode="text",
        textfont=dict(size=12, color="black"),
        showlegend=False
    ), row=1, col=1)